# Day 14 — Generator Metrics: Faithfulness & Answer Relevancy

> **Module 3 · RAG Evaluation**

Today we evaluate the **generator** of our RAG pipeline.

We already evaluated the retriever in Days 12–13. Now we ask:

1. Did the LLM use the retrieved context correctly?
2. Did the answer actually answer the user's question?

We will use two metrics:

- **Faithfulness** → Is the answer supported by the retrieved context?
- **Answer Relevancy** → Does the answer address the user's question?

In [1]:
import os

from dotenv import load_dotenv
from deepeval.models import LocalModel

load_dotenv()

os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

judge = LocalModel(
    model="gpt-4o-mini",
    base_url="https://api.openai.com/v1",
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
)

print("Judge:", judge.get_model_name())

Judge: gpt-4o-mini (Local Model)


## 14.1 — Limit concurrent requests

We use a maximum of **2 concurrent requests** to reduce the chance of hitting API rate limits.

This setting will be reused in later notebooks.

In [2]:
from deepeval.evaluate.configs import AsyncConfig

async_config = AsyncConfig(
    max_concurrent=2
)

print("Max concurrent requests:", async_config.max_concurrent)

Max concurrent requests: 2


## 14.2 — Build a small RAG pipeline

Our pipeline has two steps:

**Question → Retriever → Context → Generator → Answer**


The important part for today's evaluation is that we keep the **actual retrieved context** along with the generated answer.

In [3]:
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

CORPUS = [
    {
        "title": "Tokens",
        "content": (
            "Large language models split text into tokens, common character "
            "sequences roughly 4 characters or three-quarters of a word long. "
            "Both the prompt and the output are counted in tokens."
        ),
    },
    {
        "title": "Embeddings",
        "content": (
            "An embedding is a fixed-length vector that represents the meaning "
            "of a piece of text. Texts with similar meaning have vectors that "
            "are close together, usually measured by cosine similarity."
        ),
    },
    {
        "title": "Retrieval-Augmented Generation",
        "content": (
            "RAG grounds an LLM's answer in external documents. At query time "
            "the system retrieves the most relevant chunks and passes them to "
            "the model as context, which reduces hallucination."
        ),
    },
    {
        "title": "Hallucination",
        "content": (
            "A hallucination is fluent, confident text that is factually wrong "
            "or unsupported by its sources. Grounding answers in retrieved "
            "context is the main defense."
        ),
    },
    {
        "title": "AI agents",
        "content": (
            "An AI agent is an LLM given a goal, tools, and a loop: plan, call "
            "a tool, observe the result, and decide the next step."
        ),
    },
]


def retrieve(query: str, k: int = 2) -> list[str]:
    """Simple keyword-overlap retriever."""
    q_words = set(query.lower().split())

    scored = []

    for doc in CORPUS:
        doc_words = set(
            (doc["title"] + " " + doc["content"]).lower().split()
        )

        overlap = len(q_words & doc_words)
        scored.append((overlap, doc))

    scored.sort(key=lambda x: x[0], reverse=True)

    return [doc["content"] for _, doc in scored[:k]]

## 14.3 — Generate an answer from the retrieved context

The generator is instructed to answer **only from the retrieved context**.

This gives us something important to evaluate:

> Can the model stay grounded in the information it was given?

In [4]:
RAG_SYSTEM = (
    "Answer ONLY using the provided context. "
    "If the context does not contain the answer, say "
    "'I don't have that information.' "
    "Be concise."
)


def answer_with_openai(question: str, passages: list[str]) -> str:
    context = "\n\n".join(passages)

    prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": RAG_SYSTEM},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )

    return response.choices[0].message.content.strip()


def ask_rag(question: str, k: int = 2) -> tuple[str, list[str]]:
    passages = retrieve(question, k)
    answer = answer_with_openai(question, passages)

    return answer, passages

## 14.4 — Run the RAG

We will test both:

- questions that our corpus can answer
- a question that our corpus cannot answer

The second case is important because a good RAG should **refuse instead of inventing information** when the required information is unavailable.

In [5]:
QUESTIONS = [
    "What is a token in an LLM?",
    "How does RAG reduce hallucination?",
    "How do I containerize a model with Docker?",
]

rag_rows = []

for question in QUESTIONS:
    answer, passages = ask_rag(question)

    rag_rows.append(
        {
            "input": question,
            "actual_output": answer,
            "retrieval_context": passages,
        }
    )

    print("Q:", question)
    print("A:", answer)
    print("Retrieved passages:", len(passages))
    print("-" * 70)

Q: What is a token in an LLM?
A: I don't have that information.
Retrieved passages: 2
----------------------------------------------------------------------
Q: How does RAG reduce hallucination?
A: RAG reduces hallucination by grounding an LLM's answer in external documents, retrieving the most relevant chunks at query time, and providing them as context to the model.
Retrieved passages: 2
----------------------------------------------------------------------
Q: How do I containerize a model with Docker?
A: I don't have that information.
Retrieved passages: 2
----------------------------------------------------------------------


## 14.5 — Convert the results into DeepEval test cases

For generator evaluation, the important fields are:

- `input` → the user's question
- `actual_output` → what our RAG generated
- `retrieval_context` → what the retriever actually returned

We keep these together so DeepEval can evaluate the generator against its real context.

In [6]:
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input=row["input"],
        actual_output=row["actual_output"],
        retrieval_context=row["retrieval_context"],
    )
    for row in rag_rows
]

print("Test cases:", len(test_cases))

Test cases: 3


## 14.6 — Evaluate the generator

We will use two metrics:

**Faithfulness**
→ Is the answer supported by the retrieved context?

**Answer Relevancy**
→ Does the answer actually address the question?

These metrics evaluate different failure modes, so we use them together.

In [7]:
from deepeval import evaluate
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
)

metrics = [
    FaithfulnessMetric(
        model=judge,
        threshold=0.5,
    ),
    AnswerRelevancyMetric(
        model=judge,
        threshold=0.5,
    ),
]

results = evaluate(
    test_cases=test_cases,
    metrics=metrics,
    async_config=async_config,
)

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o-mini (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini (Local Model), strict=False, 
async_mode=True)...

c:\Users\T14s\AppData\Local\Programs\Python\Python311\Lib\site-packages\rich\live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:            What is a token in an LLM?                                                             │
│  │     Actual Output:    I don't have that information.                                                         │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Faithfulness     │ 1.00  │ 0.50      │ The score is 1.00 because there are no contradi...        │
│        FAIL  │ Answer Relevancy │ 0.00  │ 0.50      │ The score is 0.00 because the output completely fails     │
│              │                  │       │           │ to address the question about what a token is in an       │
│              │                  │       │           │ LLM, providing no relevant information.                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_2                                                                                                 │
│  ├──   Input:            How do I containerize a model with Docker?                                             │
│  │     Actual Output:    I don't have that information.                                                         │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Faithfulness     │ 1.00  │ 0.50      │ The score is 1.00 because there are no contradi...        │
│        FAIL  │ Answer Relevancy │ 0.00  │ 0.50      │ The score is 0.00 because the output fails to provide     │
│              │                  │       │           │ any relevant information on containerizing a model with   │
│              │                  │       │           │ Docker, instead indicating a lack of information, which   │
│              │                  │       │           │ does not address the question.                            │
│                                                                                                                 │
╰───────────────────────────────────────────────────────────

⚠ WARNING: No hyperparameters logged.
» ]8;id=334963;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 10.37s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 33.33% | Passed: 1 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## 14.7 — Read the evaluation results

Don't look only at the score.

The **reason** tells us why the judge gave that score. In real evaluation work, this is often more useful than the number itself.

In [8]:
for i, result in enumerate(results.test_results, start=1):
    print(f"\n{'=' * 70}")
    print(f"CASE {i}")
    print(f"Question: {result.input}")
    print(f"{'=' * 70}")

    for metric_result in result.metrics_data:
        print(
            f"{metric_result.name:<22} "
            f"score={metric_result.score:.2f} "
            f"success={metric_result.success}"
        )
        print(f"Reason: {metric_result.reason[:200]}")


CASE 1
Question: How does RAG reduce hallucination?
Faithfulness           score=1.00 success=True
Reason: The score is 1.00 because there are no contradictions, indicating that the actual output perfectly aligns with the retrieval context.
Answer Relevancy       score=1.00 success=True
Reason: The score is 1.00 because the response directly addresses the question about how RAG reduces hallucination without any irrelevant statements.

CASE 2
Question: What is a token in an LLM?
Faithfulness           score=1.00 success=True
Reason: The score is 1.00 because there are no contradictions, indicating that the actual output perfectly aligns with the retrieval context.
Answer Relevancy       score=0.00 success=False
Reason: The score is 0.00 because the output completely fails to address the question about what a token is in an LLM, providing no relevant information.

CASE 3
Question: How do I containerize a model with Docker?
Faithfulness           score=1.00 success=True
Reason: The score

## Key Takeaways

### Faithfulness

**Question:** Is the answer supported by the retrieved context?

Low faithfulness usually means the generator introduced information that was not present in the retrieved documents.

### Answer Relevancy

**Question:** Does the answer actually address the user's question?

An answer can be faithful but still irrelevant.

### The RAG evaluation picture

So far:

- **Days 12–13 → Retriever**
  - Contextual Precision
  - Contextual Recall
  - Contextual Relevancy

- **Day 14 → Generator**
  - Faithfulness
  - Answer Relevancy

This gives us a much better diagnosis than looking at one overall RAG score.

> **Retriever asks:** Did we find the right information?

> **Generator asks:** Did we use that information correctly?